In [ ]:
import gurobipy as gp
from gurobipy import GRB
import numpy as np

# Parameters and indices
n = 5  # Number of points
K = 3  # Number of products
S = 3  # Number of scenarios

np.random.seed(7)

# Distance cost matrix
C_t = 3 * np.random.rand(n, n)

# Initial shelf life, required shelf life, and quality reduction factors
SL_0 = 3000 * np.random.rand(K)  # Initial shelf life
SL_r = 100 * np.random.rand(K)  # Required shelf life at destination
Q = np.random.uniform(0.5, 1, K)  # Quality reduction factor

# Priority customers
P = np.random.choice([0, 1], size=n, p=[.7, .3])
alpha = 7

In [ ]:
# Generate scenarios for R (delays) and T (travel times)
R_scenarios = [2 * np.random.rand(n) for _ in range(S)]  # Delay at each stop for each scenario
T_scenarios = [5 * np.random.rand(n, n) for _ in range(S)]  # Travel time for each scenario

# Scenario probabilities (assuming equal probability for simplicity)
scenario_probs = [1 / S for _ in range(S)]

In [ ]:
# Create model
model = gp.Model('Stochastic Perishable Produce Transportation')

Academic license - for non-commercial use only - expires 2024-11-07
Using license file C:\Users\alike\gurobi.lic


In [ ]:
# Variables
x = model.addVars(n, n, vtype=GRB.BINARY)  # 1 if arc i-j is part of the route
u = model.addVars(n, lb=1, ub=n, vtype=GRB.CONTINUOUS)  # Auxiliary variable for subtour elimination

In [ ]:
# Constraints

# Each node is entered once
model.addConstrs((gp.quicksum(x[i, j] for i in range(n)) == 1 for j in range(n)), name='const1')

# No self-loops
model.addConstrs((x[i, j] == 0 for i in range(n) for j in range(n) if i == j), name='const2')

# Each node is exited once
model.addConstrs((gp.quicksum(x[i, j] for j in range(n)) == 1 for i in range(n)), name='const3')

# MTZ Constraints (Subtour elimination)
model.addConstrs((u[i] - u[j] + n * x[i, j] <= n for i in range(n) for j in range(n) if i != j), name='const4')

# Total in-transit time constraint for each scenario
for s in range(S):
    R = R_scenarios[s]
    T = T_scenarios[s]
    model.addConstrs(
        (gp.quicksum((R[i] + T[i, j]) * x[i, j] for i in range(n) for j in range(n)) <= Q[k] * SL_0[k] - SL_r[k] for k in range(K)),
        name=f'time_constraint_scenario_{s}'
    )

In [ ]:
# Objective function: Minimize expected cost over all scenarios
expected_cost = gp.quicksum(
    scenario_probs[s] * gp.quicksum(C_t[i, j] * x[i, j] for i in range(n) for j in range(n)) for s in range(S)
)
priority_term = gp.quicksum((P[i] + P[j]) * alpha * x[i, j] for i in range(n) for j in range(n))

# Set the objective (minimize cost minus priority term)
model.setObjective(expected_cost - priority_term, GRB.MINIMIZE)

In [ ]:
# Optimize the model
model.optimize()

Gurobi Optimizer version 9.1.1 build v9.1.1rc0 (win64)
Thread count: 2 physical cores, 4 logical processors, using up to 4 threads
Optimize a model with 44 rows, 30 columns and 340 nonzeros
Model fingerprint: 0x04f7ce45
Variable types: 5 continuous, 25 integer (25 binary)
Coefficient statistics:
  Matrix range     [6e-01, 6e+00]
  Objective range  [7e-02, 1e+01]
  Bounds range     [1e+00, 5e+00]
  RHS range        [1e+00, 2e+03]
Found heuristic solution: objective -21.9463895
Presolve removed 14 rows and 5 columns
Presolve time: 0.00s
Presolved: 30 rows, 25 columns, 100 nonzeros
Variable types: 5 continuous, 20 integer (20 binary)

Root relaxation: objective -2.336994e+01, 10 iterations, 0.00 seconds

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth IntInf | Incumbent    BestBd   Gap | It/Node Time

*    0     0               0     -23.3699367  -23.36994  0.00%     -    0s

Explored 0 nodes (10 simplex iterations) in 0.03 seconds
Threa

In [ ]:
# Display results
if model.status == GRB.OPTIMAL:
    print(f"Optimal objective value: {model.objVal}")

    for var in model.getVars():
        if var.X > 0:
            print(f"{var.VarName}: {var.X}")
else:
    print(f"Optimization was unsuccessful. Status code: {model.status}")

Optimal objective value: -23.369936747439244
C1: 1.0
C7: 1.0
C13: 1.0
C19: 1.0
C20: 1.0
C25: 1.0
C26: 1.0
C27: 1.0
C28: 1.0
C29: 1.0
